In [0]:
WITH bin_mapping_100_129 AS (
  SELECT * FROM VALUES
    ('BP TAG+$100 - 802 CR'), ('PREM TAG+$100 - 804 CR'), ('PROMO BP TAG $100 - 1002 CR'), ('TAG $100 - 790 CR'),
    ('BP TAG+$129 - 1107 CR'), ('PREM TAG+$129 - 1109 CR'), ('TAG $129 - 1095 CR')
  AS t(Item_Description)
),

conditions AS (
  SELECT * FROM VALUES
    (2026, 'Offer', DATE'2026-05-18', DATE'2026-09-07'),
    (2026, 'Pre',   DATE'2026-04-13', DATE'2026-05-17'),
    (2025, 'Offer', DATE'2025-05-18', DATE'2025-09-07'),
    (2025, 'Pre',   DATE'2025-04-13', DATE'2025-05-17')
  AS c(Year, Period_Type, date_start, date_end)
),

initial_purchases AS (
  SELECT
    c.card_barcode,
    MIN(CAST(c.transaction_datetime AS DATE)) AS purchase_date,
    cond.Year,
    cond.Period_Type
  FROM cpx_dataanalytics_silver.embed.cardhistory_result c
  INNER JOIN bin_mapping_100_129 bm ON bm.Item_Description = c.description
  INNER JOIN conditions cond ON CAST(c.transaction_datetime AS DATE) BETWEEN cond.date_start AND cond.date_end
  WHERE c.transaction_type = 'SALECP'
  GROUP BY c.card_barcode, cond.Year, cond.Period_Type
),

revisits AS (
  SELECT
    p.card_barcode,
    p.Year,
    p.Period_Type,
    COUNT(DISTINCT CAST(c.transaction_datetime AS DATE)) AS Revisit_Days
  FROM initial_purchases p
  INNER JOIN conditions cond ON cond.Year = p.Year AND cond.Period_Type = p.Period_Type
  INNER JOIN cpx_dataanalytics_silver.embed.cardhistory_result c
    ON c.card_barcode = p.card_barcode
    AND CAST(c.transaction_datetime AS DATE) > p.purchase_date
  WHERE c.transaction_type IN ('SALECP', 'DFREE')
    AND CAST(c.transaction_datetime AS DATE) BETWEEN cond.date_start AND cond.date_end
  GROUP BY p.card_barcode, p.Year, p.Period_Type
)

SELECT
  p.Year,
  p.Period_Type,
  COUNT(DISTINCT p.card_barcode) AS Total_Gamebands,
  COUNT(DISTINCT r.card_barcode) AS Gamebands_With_Revisit,
  COALESCE(SUM(r.Revisit_Days), 0) AS Total_Revisit_Days
FROM initial_purchases p
LEFT JOIN revisits r ON r.card_barcode = p.card_barcode AND r.Year = p.Year AND r.Period_Type = p.Period_Type
GROUP BY p.Year, p.Period_Type
ORDER BY p.Year, p.Period_Type DESC;